# 서울 열린데이터 API 데이터 수집

서울 열린데이터광장 OpenAPI를 통해 **지하철·버스 시간대별 승하차 데이터**를 수집한다.

## 사용 데이터셋
- **지하철**: 서울시 지하철호선별 역별 시간대별 승하차 인원 정보 (OA-12913)
- **버스**: 서울시 버스노선별 정류장별 시간대별 승하차 인원 정보 (OA-12252)

## 1. 사전 준비

### 1.1 인증키 발급
[서울 열린데이터광장](https://data.seoul.go.kr)에서 회원가입 후 **데이터셋별로** 인증키 발급.
- 지하철 데이터셋(OA-12913)에서 인증키 신청 → `SEOUL_API_KEY_1`
- 버스 데이터셋(OA-12252)에서 인증키 신청 → `SEOUL_API_KEY_2`

### 1.2 `.env` 파일 생성
프로젝트 루트에 `.env` 파일을 만들고 아래 내용 입력:
```
SEOUL_API_KEY_1=지하철용_인증키
SEOUL_API_KEY_2=버스용_인증키
```

> 만약 키 배정이 반대로 되어 있다면 `.env`의 두 값만 서로 바꾸면 됩니다.

### 1.3 패키지 설치

In [ ]:
%pip install -q requests python-dotenv

## 2. 임포트 및 인증키 로드

In [ ]:
import csv
import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv()

API_KEY_SUBWAY = os.environ.get("SEOUL_API_KEY_1")
API_KEY_BUS    = os.environ.get("SEOUL_API_KEY_2")

for name, value in [("SEOUL_API_KEY_1", API_KEY_SUBWAY), ("SEOUL_API_KEY_2", API_KEY_BUS)]:
    if not value:
        raise RuntimeError(f"{name} 환경변수가 설정되지 않았습니다. .env 파일을 확인하세요.")
    print(f"{name} loaded: {value[:6]}…")

## 3. 설정값

⚠️ **서비스명**은 각 데이터셋 페이지의 **[Open API] 탭**에서 확인 가능:
- OA-12913 (지하철): https://data.seoul.go.kr/dataList/OA-12913/A/1/datasetView.do
- OA-12252 (버스): https://data.seoul.go.kr/dataList/OA-12252/A/1/datasetView.do

샘플 URL의 `/{서비스명}/` 부분이 실제 서비스명. 다르면 `DATASETS` 딕셔너리의 `service` 값만 교체.

In [ ]:
BASE_URL = "http://openapi.seoul.go.kr:8088"
PAGE_SIZE = 1000  # 서울 OpenAPI 최대 1,000건/요청
SLEEP_SEC = 0.2   # rate limit 보호

DATASETS = {
    "subway": {
        "service": "CardSubwayTime",
        "prefix":  "subway",
        "api_key": API_KEY_SUBWAY,
    },
    "bus": {
        "service": "CardBusTimeNew",
        "prefix":  "bus",
        "api_key": API_KEY_BUS,
    },
}

## 4. 헬퍼 함수

### 4.1 단일 페이지 요청

In [ ]:
def fetch_page(service: str, api_key: str, start: int, end: int, *extra: str) -> dict:
    """단일 페이지 요청. 서울 API 래퍼 body를 반환."""
    parts = [BASE_URL, api_key, "json", service, str(start), str(end), *extra]
    url = "/".join(parts)

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    body = payload.get(service)
    if body is None:
        err = payload.get("RESULT") or payload.get("result") or payload
        raise RuntimeError(f"API error for {service}: {err}")

    code = (body.get("RESULT") or {}).get("CODE", "")
    if code and not code.startswith("INFO-"):
        raise RuntimeError(f"API error for {service}: {body['RESULT']}")

    return body

### 4.2 페이지네이션 자동 처리

In [ ]:
def fetch_all(service: str, api_key: str, *extra: str):
    """모든 row를 yield. list_total_count까지 페이지 순회."""
    start = 1
    while True:
        end = start + PAGE_SIZE - 1
        body = fetch_page(service, api_key, start, end, *extra)
        rows = body.get("row") or []
        if not rows:
            return
        yield from rows

        total = body.get("list_total_count", 0)
        if end >= total:
            return
        start = end + 1
        time.sleep(SLEEP_SEC)

### 4.3 CSV 저장 및 수집 함수

In [ ]:
def save_csv(rows: list, path: Path) -> int:
    if not rows:
        return 0
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    return len(rows)


def collect(dataset_key: str, use_month: str, out_dir: Path = Path("data/raw")) -> None:
    meta = DATASETS[dataset_key]
    out_path = out_dir / f"{meta['prefix']}_{use_month}.csv"

    print(f"[{dataset_key}/{use_month}] fetching {meta['service']}…")
    rows = list(fetch_all(meta["service"], meta["api_key"], use_month))
    n = save_csv(rows, out_path)
    if n == 0:
        print(f"  ⚠ no rows for {out_path.name}")
        return
    size_mb = out_path.stat().st_size / (1024 * 1024)
    print(f"  saved {n:,} rows ({size_mb:.1f} MB) → {out_path}")

## 5. 데이터 수집 실행

### 5.1 단일 월 테스트
먼저 한 달치만 받아서 정상 동작하는지 확인.

In [ ]:
# 지하철·버스 각각 1개월 테스트
collect("subway", "202505")
collect("bus",    "202505")

### 5.2 여러 월 일괄 수집

**누적 100MB 이상** 확보를 위해 25년 1월 ~ 26년 5월(총 17개월) 수집.

> ⏱ 17개월 × 2 데이터셋 → API 호출 + 페이지네이션 포함 약 15~30분 소요.

In [ ]:
# 25년 1월 ~ 26년 5월 (17개월)
months = (
    [f"2025{m:02d}" for m in range(1, 13)]   # 2025년 1월~12월
    + [f"2026{m:02d}" for m in range(1, 6)]  # 2026년 1월~5월
)
out_dir = Path("data/raw")

print(f"수집 대상: {len(months)}개월 × 2 데이터셋 = {len(months) * 2}건\n")

for ym in months:
    for ds in ["subway", "bus"]:
        try:
            collect(ds, ym, out_dir)
        except Exception as e:
            print(f"  ERROR [{ds}/{ym}]: {e}")

### 5.3 수집 결과 확인
총 누적 용량이 100MB 이상인지 확인.

In [ ]:
total_size = 0.0
for f in sorted(Path("data/raw").glob("*.csv")):
    size_mb = f.stat().st_size / (1024 * 1024)
    total_size += size_mb
    print(f"  {f.name}: {size_mb:.1f} MB")
print(f"\n총 누적: {total_size:.1f} MB")